# GLENGlat submission preparation notebook

This notebook prepares englacial temperature data for submission to **GLENGlat** (Global Englacial Temperature database).

It is tailored to the data layout used in this project:
- hourly time series from the Tynitag and Geoprecision sensors
- borehole metadata from the study-site / thickness-mapping workflow
- mostly ablation-area boreholes, with AH1G and AH3G in accumulation

The notebook builds submission-ready `borehole.csv` and `measurement.csv` files and then validates them with the GLENGlat Frictionless workflow before submission.

## Source inventory

The notebook pulls from three project-specific sources:
- hourly temperature files from `NTC_tynitag/temperature_data/full_timeseries/`
- hourly temperature files from `thermistor_chains/temperature_data/`
- borehole positions, elevations, and depth settings from `thermistor_borehole_settings/`

The paths below mirror the analysis notebooks so the submission prep stays in sync with the existing workflow.

In [13]:
from pathlib import Path
import os
import sys
import subprocess

import numpy as np
import pandas as pd
import geopandas as gpd

# Add project root to Python path
project_root = Path('/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from processing.gpr_processing import load_borehole_positions
from processing.thermistor_processing import ThermistorData, read_thermistor_depths

fieldwork_root = Path('/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/data/fieldwork_data/THERMAP_2024_2025/icetemperature_data')

ntc_root = fieldwork_root / 'NTC_tynitag' / 'temperature_data' / 'full_timeseries'
gp_root = fieldwork_root / 'thermistor_chains' / 'temperature_data'
settings_root = fieldwork_root / 'thermistor_borehole_settings'
coords_csv = settings_root / 'thermistor_coordinates.csv'

output_dir = project_root / 'new_data' / 'glenglat_submission'
output_dir.mkdir(parents=True, exist_ok=True)

print('project_root:', project_root)
print('ntc_root:', ntc_root)
print('gp_root:', gp_root)
print('settings_root:', settings_root)
print('coords_csv:', coords_csv)
print('output_dir:', output_dir)

project_root: /Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers
ntc_root: /Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/data/fieldwork_data/THERMAP_2024_2025/icetemperature_data/NTC_tynitag/temperature_data/full_timeseries
gp_root: /Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/data/fieldwork_data/THERMAP_2024_2025/icetemperature_data/thermistor_chains/temperature_data
settings_root: /Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/data/fieldwork_data/THERMAP_2024_2025/icetemperature_data/thermistor_borehole_settings
coords_csv: /Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/data/fieldwork_data/THERMAP_2024_2025/icetemperature_data/thermistor_borehole_settings/thermistor_coordinates.csv
output_dir: /Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/new_data/glenglat_submission


## Source maps and helpers

The next cells define the hourly sensor files, depth metadata, and small helper functions used to build GLENGlat tables.

Borehole metadata are read from the same coordinate table used in the study-site maps, and all boreholes are tagged as `ablation`.

In [14]:
def pick_first_existing(df, candidates, default=None):
    column_lookup = {str(column).strip().lower(): column for column in df.columns}
    for name in candidates:
        key = str(name).strip().lower()
        if key in column_lookup:
            return column_lookup[key]
    return default


def as_clean_string(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    return pd.NA if text == '' else text


def timestamp_parts(ts):
    timestamp = pd.Timestamp(ts)
    return timestamp.strftime('%Y-%m-%d'), timestamp.strftime('%H:%M:%S')


SOURCE_SPECS = [
    {'label': 'AH1G', 'glacier_name': 'Alphubel South', 'family': 'geoprecision',
     'raw_file': gp_root / 'A551FE/raw/A551FE_20250916075641.csv',
     'depth_file': settings_root / 'thermistor_settings_ah1g.csv',
     'logger_id': None, 'raw_kind': 'chain'},
    {'label': 'AH2G', 'glacier_name': 'Alphubel South', 'family': 'geoprecision',
     'raw_file': gp_root / 'A55204/raw/A55204_20250916082002.csv',
     'depth_file': settings_root / 'thermistor_settings_ah2g.csv',
     'logger_id': None, 'raw_kind': 'chain'},
    {'label': 'AH3G', 'glacier_name': 'Alphubel South', 'family': 'geoprecision',
     'raw_file': gp_root / 'A55205/raw/A55205_20250916074654.csv',
     'depth_file': settings_root / 'thermistor_settings_ah3g.csv',
     'logger_id': None, 'raw_kind': 'chain'},
    {'label': 'HS1G', 'glacier_name': 'Hohsaasgletscher', 'family': 'geoprecision',
     'raw_file': gp_root / 'A551FD/raw/A551FD_20250927153221.csv',
     'depth_file': settings_root / 'thermistor_settings_hs1g.csv',
     'logger_id': None, 'raw_kind': 'chain'},
    {'label': 'HS2G', 'glacier_name': 'Hohsaasgletscher', 'family': 'geoprecision',
     'raw_file': gp_root / 'A55203/raw/A55203_20250927151514.csv',
     'depth_file': settings_root / 'thermistor_settings_hs2g.csv',
     'logger_id': None, 'raw_kind': 'chain'},
    {'label': 'HS3G', 'glacier_name': 'Hohsaasgletscher', 'family': 'geoprecision',
     'raw_file': gp_root / 'A55200/raw/A55200_20250927151301.csv',
     'depth_file': settings_root / 'thermistor_settings_hs3g.csv',
     'logger_id': None, 'raw_kind': 'chain'},
    {'label': 'CJ1G', 'glacier_name': 'Chessjengletscher', 'family': 'geoprecision',
     'raw_file': gp_root / 'A55201/raw/A55201_20251215143911.csv',
     'depth_file': settings_root / 'thermistor_settings_cj1g.csv',
     'logger_id': None, 'raw_kind': 'chain'},
    {'label': 'CJ2G', 'glacier_name': 'Chessjengletscher', 'family': 'geoprecision',
     'raw_file': gp_root / 'A55202/raw/A55202_20251215144715.csv',
     'depth_file': settings_root / 'thermistor_settings_cj2g.csv',
     'logger_id': None, 'raw_kind': 'chain'},
    {'label': 'AH1TT', 'glacier_name': 'Alphubel South', 'family': 'tynitag',
     'raw_file': ntc_root / 'AH1TT_20250322_20250916.csv',
     'depth_file': settings_root / 'thermistor_settings_ah1tt.csv',
     'logger_id': '9', 'raw_kind': 'ntc'},
    {'label': 'AH2TT', 'glacier_name': 'Alphubel South', 'family': 'tynitag',
     'raw_file': ntc_root / 'AH2TT_20240821_20250916_spliced.csv',
     'depth_file': settings_root / 'thermistor_settings_ah2tt.csv',
     'logger_id': '10', 'raw_kind': 'ntc'},
    {'label': 'AH3TT', 'glacier_name': 'Alphubel South', 'family': 'tynitag',
     'raw_file': ntc_root / 'AH3TT_20250806_20250916.csv',
     'depth_file': settings_root / 'thermistor_settings_ah3tt.csv',
     'logger_id': '13', 'raw_kind': 'ntc'},
    {'label': 'CJ1TT', 'glacier_name': 'Chessjengletscher', 'family': 'tynitag',
     'raw_file': ntc_root / 'CJ1TT_20240809_20250808_spliced.csv',
     'depth_file': settings_root / 'thermistor_settings_cj1tt.csv',
     'logger_id': '7', 'raw_kind': 'ntc'},
    {'label': 'CJ2TT', 'glacier_name': 'Chessjengletscher', 'family': 'tynitag',
     'raw_file': ntc_root / 'CJ2TT_20240809_20250808_spliced.csv',
     'depth_file': settings_root / 'thermistor_settings_cj2tt.csv',
     'logger_id': '8', 'raw_kind': 'ntc'},
    {'label': 'CJ3TT', 'glacier_name': 'Chessjengletscher', 'family': 'tynitag',
     'raw_file': ntc_root.parent / '2024_2025' / 'new_naming' / 'CJ3TT_20251215.csv',
     'depth_file': settings_root / 'thermistor_settings_cj3tt.csv',
     'logger_id': '14', 'raw_kind': 'ntc'},
    {'label': 'CJ4TT', 'glacier_name': 'Chessjengletscher', 'family': 'tynitag',
     'raw_file': ntc_root.parent / '2024_2025' / 'new_naming' / 'CJ4TT_20251215.csv',
     'depth_file': settings_root / 'thermistor_settings_cj4tt.csv',
     'logger_id': '15', 'raw_kind': 'ntc'},
    {'label': 'HS1TT', 'glacier_name': 'Hohsaasgletscher', 'family': 'tynitag',
     'raw_file': ntc_root / 'HS1TT_20240808_20250927_spliced.csv',
     'depth_file': settings_root / 'thermistor_settings_hs1tt.csv',
     'logger_id': '5', 'raw_kind': 'ntc'},
    {'label': 'HS2TT', 'glacier_name': 'Hohsaasgletscher', 'family': 'tynitag',
     'raw_file': ntc_root / 'HS2TT_20240808_20250927_spliced.csv',
     'depth_file': settings_root / 'thermistor_settings_hs2tt.csv',
     'logger_id': '6', 'raw_kind': 'ntc'},
    {'label': 'SR1TT', 'glacier_name': 'Glacier du Sex Rouge', 'family': 'tynitag',
     'raw_file': ntc_root / 'SR1TT_20240806_20250724_spliced.csv',
     'depth_file': settings_root / 'thermistor_settings_sr1tt.csv',
     'logger_id': '1', 'raw_kind': 'ntc'},
    {'label': 'SR2TT', 'glacier_name': 'Glacier du Sex Rouge', 'family': 'tynitag',
     'raw_file': ntc_root / 'SR2TT_20240806_20250724_spliced.csv',
     'depth_file': settings_root / 'thermistor_settings_sr2tt.csv',
     'logger_id': '2', 'raw_kind': 'ntc'},
    {'label': 'GT1TT', 'glacier_name': 'Glacier de Tortin', 'family': 'tynitag',
     'raw_file': ntc_root / 'GT1TT_20240807_20250723_spliced.csv',
     'depth_file': settings_root / 'thermistor_settings_gt1tt.csv',
     'logger_id': None, 'raw_kind': 'ntc'},
    {'label': 'GT2TT', 'glacier_name': 'Glacier de Tortin', 'family': 'tynitag',
     'raw_file': ntc_root / 'GT2TT_20240807_20250723_spliced.csv',
     'depth_file': settings_root / 'thermistor_settings_gt2tt.csv',
     'logger_id': None, 'raw_kind': 'ntc'},
    {'label': 'CV1TT', 'glacier_name': 'Vadret dal Corvatsch', 'family': 'tynitag',
     'raw_file': ntc_root / 'CT1TT_20240828_20250905.csv',
     'depth_file': settings_root / 'thermistor_settings_cv1tt.csv',
     'logger_id': '12', 'raw_kind': 'ntc'},
    {'label': 'CV2TT', 'glacier_name': 'Vadret dal Corvatsch', 'family': 'tynitag',
     'raw_file': ntc_root / 'CT2TT_20240828_20250905_spliced.csv',
     'depth_file': settings_root / 'thermistor_settings_cv2tt.csv',
     'logger_id': '12', 'raw_kind': 'ntc'},
]

SOURCE_LABELS = [spec['label'] for spec in SOURCE_SPECS]


# Load calibration offsets once
_CHAIN_OFFSETS_PATH = project_root / 'new_data' / 'calibration' / 'corrected_chain_offsets.csv'
_NTC_OFFSETS_PATH   = fieldwork_root / 'NTC_tynitag' / 'calibration_data' / 'all_logger_offsets.csv'
chain_offsets_df = pd.read_csv(_CHAIN_OFFSETS_PATH, index_col='chain')
ntc_offsets_df   = pd.read_csv(_NTC_OFFSETS_PATH)


def load_source_temperatures(spec):
    raw_file = Path(spec['raw_file'])
    if not raw_file.exists():
        print(f"WARNING: missing raw file for {spec['label']}: {raw_file}")
        return pd.DataFrame()

    if spec['raw_kind'] == 'chain':
        # chain ID is the grandparent directory name (e.g. A551FE/raw/file.csv)
        chain_id = raw_file.parts[-3]
        if chain_id in chain_offsets_df.index:
            offsets = chain_offsets_df.loc[chain_id].dropna().to_dict()
            data = ThermistorData(str(raw_file), ',').get_chain_data_with_offsets(offsets=offsets)
        else:
            print(f"WARNING: no chain offsets for {chain_id} ({spec['label']}), using raw data")
            data = ThermistorData(str(raw_file), ',').get_chain_data()
    else:
        if raw_file.name.endswith('_spliced.csv'):
            # spliced files are already calibrated — do not apply offsets again
            data = ThermistorData(str(raw_file), ',').get_ntc_data()
        else:
            logger_id = spec.get('logger_id')
            if logger_id is not None:
                data = ThermistorData(str(raw_file), ',').get_ntc_data_with_offsets(logger_id, ntc_offsets_df)
            else:
                print(f"WARNING: no logger_id for {spec['label']}, using uncalibrated raw data")
                data = ThermistorData(str(raw_file), ',').get_ntc_data()

    if data.empty:
        print(f"WARNING: empty data for {spec['label']}")
        return data

    return data


def load_sensor_depths(spec, when=None):
    """Return sensor depths from the settings file.

    Pass when=drill_date to get Z_init (depths at installation).
    Pass when=None (default) to get Z_final (last recorded, ablation-corrected).
    """
    depth_file = Path(spec['depth_file'])
    if not depth_file.exists():
        print(f"WARNING: missing depth file for {spec['label']}: {depth_file}")
        return {}
    return read_thermistor_depths(str(depth_file), when=when)


def raw_file_name(spec):
    return Path(spec['raw_file']).name


_EQUILIBRIUM_NOTE = "Thermal equilibration typically reached 2-3 days post-installation."


def expand_ntc_timeseries(spec, data, depths):
    records = []
    temp_cols = {
        'white probe': 'White Probe Temperature',
        'black probe': 'Black Probe Temperature',
    }
    for _, row in data.iterrows():
        timestamp = pd.Timestamp(row['TIME'])
        date_str, time_str = timestamp_parts(timestamp)
        for sensor_name, temp_col in temp_cols.items():
            if temp_col not in row.index:
                continue
            temp = row[temp_col]
            depth = depths.get(sensor_name)
            if pd.isna(temp) or depth is None or pd.isna(depth):
                continue
            records.append({
                'label': spec['label'],
                'glacier_name': spec['glacier_name'],
                'source_family': spec['family'],
                'sensor_name': sensor_name,
                'depth': float(depth),
                'temperature': float(temp),
                'date_min': date_str,
                'date_max': date_str,
                'time': time_str,
                'utc_offset': 0,
                'equilibrium': 'false',
                'notes': f"Hourly Tynitag logger data from {raw_file_name(spec)}; {_EQUILIBRIUM_NOTE}",
            })
    return records


def expand_chain_timeseries(spec, data, depths):
    records = []
    chain_cols = [col for col in data.columns if str(col).startswith('#')]
    for _, row in data.iterrows():
        timestamp = pd.Timestamp(row['TIME'])
        date_str, time_str = timestamp_parts(timestamp)
        for chain_col in chain_cols:
            temp = row[chain_col]
            depth = depths.get(chain_col)
            if pd.isna(temp) or depth is None or pd.isna(depth):
                continue
            records.append({
                'label': spec['label'],
                'glacier_name': spec['glacier_name'],
                'source_family': spec['family'],
                'sensor_name': chain_col,
                'depth': float(depth),
                'temperature': float(temp),
                'date_min': date_str,
                'date_max': date_str,
                'time': time_str,
                'utc_offset': 0,
                'equilibrium': 'false',
                'notes': f"Hourly Geoprecision chain data from {raw_file_name(spec)}; {_EQUILIBRIUM_NOTE}",
            })
    return records


## Build borehole and measurement tables

This cell builds both GLENGlat tables from the hourly source series. Borehole metadata are enriched with fixed total borehole depth, bed contact flag, drill method, uncertainty, investigators, and GLIMS IDs.

In [15]:
elevation_col_candidates = ['elevation', 'altitude', 'z', 'height', 'elev', 'elevation_m', 'z_m']
borehole_columns = [
    'id', 'glacier_name', 'glims_id', 'latitude', 'longitude', 'elevation',
    'mass_balance_area', 'label', 'date_min', 'date_max', 'drill_method',
    'ice_depth', 'depth', 'to_bed', 'temperature_uncertainty', 'notes',
    'investigators', 'funding',
]

measurement_columns = [
    'borehole_id', 'depth', 'temperature', 'date_min', 'date_max', 'time',
    'utc_offset', 'equilibrium', 'notes', 'source_family', 'sensor_name', 'label',
]

glims_by_glacier = {
    'Alphubel South': 'G007867E46057N',
    'Chessjengletscher': 'G007919E46068N',
    'Hohsaasgletscher': 'G008002E46138N',
    'Glacier du Sex Rouge': 'G007212E46327N',
    'Glacier de Tortin': 'G007312E46081N',
    'Vadret dal Corvatsch': 'G009822E46416N',
}

mass_balance_by_label = {
    'AH1G': 'accumulation',
    'AH3G': 'accumulation',
}

# Drilling dates (= installation day); used for both date_min and date_max in borehole.csv.
# AH3TT start inferred from raw filename (AH3TT_20250806_...); CJ3TT/CJ4TT co-drilled
# with CJ1G/CJ2G campaign on 2025-08-07.
drill_date_by_label = {
    'AH1TT': '2024-08-21', 'AH2TT': '2024-08-21',
    'AH3TT': '2025-08-06',
    'CJ1TT': '2024-08-09', 'CJ2TT': '2024-08-09',
    'CJ3TT': '2025-08-08', 'CJ4TT': '2025-08-08',
    'HS1TT': '2024-08-08', 'HS2TT': '2024-08-08',
    'SR1TT': '2024-08-06', 'SR2TT': '2024-08-06',
    'GT1TT': '2024-08-07', 'GT2TT': '2024-08-07',
    'CV1TT': '2024-08-28', 'CV2TT': '2024-08-28',
    'AH1G': '2025-08-05', 'AH2G': '2025-08-06', 'AH3G': '2025-08-06',
    'CJ1G': '2025-08-07', 'CJ2G': '2025-08-07',
    'HS1G': '2025-08-13', 'HS2G': '2025-08-13', 'HS3G': '2025-08-13',
}

# Keep a separate alias for the pre-deployment filter (unchanged logic)
date_min_override_by_label = drill_date_by_label

# Manual installation datetimes for geoprecision chain boreholes (UTC).
# Times provided as approximate local Swiss summer time (CEST = UTC+2); converted here
# by subtracting 2 hours.
INSTALL_DATETIMES_UTC = {
    'AH1G': pd.Timestamp('2025-08-05 12:00:00'),  # 14:00 CEST
    'AH2G': pd.Timestamp('2025-08-06 11:00:00'),  # 13:00 CEST
    'AH3G': pd.Timestamp('2025-08-06 09:00:00'),  # 11:00 CEST
    'HS1G': pd.Timestamp('2025-08-13 10:00:00'),  # 12:00 CEST
    'HS2G': pd.Timestamp('2025-08-13 10:00:00'),  # 12:00 CEST
    'HS3G': pd.Timestamp('2025-08-13 11:00:00'),  # 13:00 CEST
    'CJ1G': pd.Timestamp('2025-08-07 11:00:00'),  # 13:00 CEST
    'CJ2G': pd.Timestamp('2025-08-07 13:00:00'),  # 15:00 CEST
}

# Total borehole depth provided from field overview table.
total_depth_by_label = {
    'AH1G': 50.6, 'AH2G': 20.3, 'AH3G': 58.3,
    'AH1TT': 15.0, 'AH2TT': 14.0, 'AH3TT': 17.0,
    'CJ1G': 38.3, 'CJ2G': 17.0,
    'CJ1TT': 11.8, 'CJ2TT': 13.5, 'CJ3TT': 20.0, 'CJ4TT': 20.0,
    'HS1G': 29.0, 'HS2G': 21.5, 'HS3G': 21.5,
    'HS1TT': 15.6, 'HS2TT': 15.4,
    'SR1TT': 15.3, 'SR2TT': 14.5,
    'GT1TT': 9.2, 'GT2TT': 15.25,
    'CV1TT': 7.0, 'CV2TT': 9.3,
}

to_bed_by_label = {
    'AH1G': 'True', 'AH2G': 'True', 'AH3G': 'True',
    'AH1TT': 'False', 'AH2TT': 'False', 'AH3TT': 'False',
    'CJ1G': 'True', 'CJ2G': 'True',
    'CJ1TT': 'False', 'CJ2TT': 'False', 'CJ3TT': 'False', 'CJ4TT': 'False',
    'HS1G': 'True', 'HS2G': 'True', 'HS3G': 'False',
    'HS1TT': 'False', 'HS2TT': 'False',
    'SR1TT': 'False', 'SR2TT': 'False',
    'GT1TT': 'False', 'GT2TT': 'False',
    'CV1TT': 'True', 'CV2TT': 'True',
}

# Snow present at surface at installation time (cm). All boreholes were drilled into bare
# ice; depth datum is the ice surface (ice_depth=0 for all boreholes).
snow_at_surface_cm = {
    'AH1G':  14,
    'AH3G': 126,
    'CJ3TT': 73,
    'CV1TT': 48,
}

coords_gdf, coords_missing = load_borehole_positions(str(coords_csv), keep_names=SOURCE_LABELS)
coords_gdf = coords_gdf.to_crs(4326)
coords_gdf['name'] = coords_gdf['name'].astype(str).str.strip()
coords_lookup = coords_gdf.set_index(coords_gdf['name'].str.upper())

raw_coords = pd.read_csv(coords_csv, sep=None, engine='python', dtype=str)
raw_coords.columns = [str(c).strip().lstrip('﻿') for c in raw_coords.columns]
name_col = pick_first_existing(raw_coords, ['name', 'station', 'site'])
if name_col is None:
    raise ValueError(f'Could not find a borehole name column in {coords_csv}. Found columns: {raw_coords.columns.tolist()}')

elevation_col = pick_first_existing(raw_coords, elevation_col_candidates)
if elevation_col is None:
    for c in raw_coords.columns:
        cl = c.strip().lower()
        if ('elev' in cl) or (cl.startswith('z')) or ('alt' in cl) or ('height' in cl):
            elevation_col = c
            break
if elevation_col is None:
    raise ValueError(
        f'Could not find an elevation column in {coords_csv}. '
        f'Found columns: {raw_coords.columns.tolist()}'
    )

raw_coords[name_col] = raw_coords[name_col].astype(str).str.strip()
raw_coords[elevation_col] = pd.to_numeric(
    raw_coords[elevation_col].astype(str).str.replace("'", '').str.replace(',', '.', regex=False),
    errors='coerce'
)
elevation_lookup = raw_coords.set_index(raw_coords[name_col].str.upper())[elevation_col]

if coords_missing:
    print(f'WARNING: missing borehole names in coordinate table: {coords_missing}')

borehole_rows = []
missing_boreholes = []
for spec in SOURCE_SPECS:
    label = spec['label']
    key = label.upper()
    if key not in coords_lookup.index:
        missing_boreholes.append(label)
        continue

    row_wgs84 = coords_lookup.loc[key]
    elev = elevation_lookup.get(key, np.nan)

    # For NTC sensors: Z_init = depths at installation date, Z_final = last recorded.
    # Ablation is derived from Z_init - Z_final of the deeper (black) probe.
    # For geoprecision chains: no ablation correction available; use current depths.
    drill_date = drill_date_by_label.get(label)
    if spec['raw_kind'] == 'ntc' and drill_date:
        initial_depths = load_sensor_depths(spec, when=drill_date)
        final_depths = load_sensor_depths(spec)
        deeper = 'black probe' if 'black probe' in initial_depths else 'white probe'
        if deeper in initial_depths and deeper in final_depths:
            ablation = round(initial_depths[deeper] - final_depths[deeper], 2)
        else:
            ablation = None
        depths = initial_depths
    else:
        depths = load_sensor_depths(spec)
        ablation = None

    depth_values = [float(v) for v in depths.values() if v is not None and not pd.isna(v)]
    inferred_depth = max(depth_values) if depth_values else pd.NA

    if label.endswith('TT'):
        uncertainty = 0.2
    elif label.endswith('G'):
        uncertainty = 0.05
    else:
        uncertainty = pd.NA

    note_parts = [
        f"Hourly englacial temperature logger ({spec['family']}); raw file {raw_file_name(spec)}",
        "Sensor depth defined relative to ice surface at time of installation (ice_depth=0)",
    ]
    if label in snow_at_surface_cm:
        note_parts.append(
            f"Snow present at surface at installation: {snow_at_surface_cm[label]} cm "
            f"(depth datum is top of ice, not top of snow)"
        )
    if ablation is not None and ablation > 0:
        note_parts.append(
            f"Estimated surface ablation during measurement period: {ablation} m "
            f"(derived from change in sensor depth, Z_init - Z_final; "
            f"reported depths are not corrected for ablation)"
        )
    if label == 'AH1TT':
        note_parts.append(
            "No data recorded between installation (2024-08-21) and first measurement "
            "(2025-03-22) due to missing logger data."
        )

    borehole_rows.append({
        'glacier_name': spec['glacier_name'],
        'glims_id': glims_by_glacier.get(spec['glacier_name'], pd.NA),
        'latitude': float(row_wgs84.geometry.y),
        'longitude': float(row_wgs84.geometry.x),
        'elevation': float(elev) if pd.notna(elev) else pd.NA,
        'mass_balance_area': mass_balance_by_label.get(label, 'ablation'),
        'label': label,
        'date_min': pd.NA,
        'date_max': pd.NA,
        'drill_method': 'thermal',
        'ice_depth': 0,
        'depth': total_depth_by_label.get(label, inferred_depth),
        'to_bed': to_bed_by_label.get(label, 'False'),
        'temperature_uncertainty': uncertainty,
        'notes': '; '.join(note_parts),
        'investigators': 'Janosch Beer (ORCID: 0009-0002-1128-5470; VAW Glaciology, ETH Zurich)',
        'funding': "ETH Zurich; Centre de recherche sur l'environnement alpin (CREALP)",
    })

if missing_boreholes:
    raise ValueError(f'No coordinates found for: {missing_boreholes}')

borehole_df = pd.DataFrame(borehole_rows)
borehole_df.insert(0, 'id', range(1, len(borehole_df) + 1))
borehole_df = borehole_df[borehole_columns]

required_mask = borehole_df[['glacier_name', 'latitude', 'longitude', 'elevation', 'mass_balance_area', 'label', 'depth']].isna().any(axis=1)
if required_mask.any():
    missing_labels = borehole_df.loc[required_mask, 'label'].tolist()
    raise ValueError(f'Missing required borehole values for: {missing_labels}')

borehole_id_map = dict(zip(borehole_df['label'], borehole_df['id']))

measurement_rows = []
for spec in SOURCE_SPECS:
    source_data = load_source_temperatures(spec)
    if source_data.empty:
        continue

    # NTC Tynitag: use Z_init depths (sensor position at installation).
    # Geoprecision chains: use current depths from settings file.
    drill_date = drill_date_by_label.get(spec['label'])
    if spec['raw_kind'] == 'ntc' and drill_date:
        depths = load_sensor_depths(spec, when=drill_date)
    else:
        depths = load_sensor_depths(spec)

    if spec['raw_kind'] == 'ntc':
        records = expand_ntc_timeseries(spec, source_data, depths)
    else:
        records = expand_chain_timeseries(spec, source_data, depths)

    for rec in records:
        rec['borehole_id'] = borehole_id_map.get(rec['label'], pd.NA)
        measurement_rows.append(rec)

measurement_df = pd.DataFrame(measurement_rows)
if measurement_df.empty:
    raise ValueError('No measurement records were built from SOURCE_SPECS.')

measurement_df = measurement_df[measurement_columns]
measurement_df['utc_offset'] = 0
measurement_df['equilibrium'] = 'false'  # default; overwritten below by 3-day threshold
measurement_df['date_min'] = pd.to_datetime(measurement_df['date_min'], errors='coerce').dt.strftime('%Y-%m-%d')
measurement_df['date_max'] = pd.to_datetime(measurement_df['date_max'], errors='coerce').dt.strftime('%Y-%m-%d')

# For geoprecision chain loggers only: drop non-hourly timestamps produced at the
# exact moment of startup/connection before the hourly schedule begins.
# Tynitag NTC loggers are excluded — their timestamps are not exactly on the hour.
chain_mask = measurement_df['source_family'] == 'geoprecision'
hourly_mask = ~chain_mask | measurement_df['time'].str.endswith(':00:00')
n_dropped_nonhourly = (~hourly_mask).sum()
measurement_df = measurement_df[hourly_mask].reset_index(drop=True)
print(f'Dropped {n_dropped_nonhourly} non-hourly geoprecision measurement rows.')

# Filter measurements to on-glacier period (exclude pre-deployment data in raw files)
_deploy_dates = measurement_df['label'].map(
    lambda lbl: date_min_override_by_label.get(lbl, '1900-01-01')
)
_keep = pd.to_datetime(measurement_df['date_min']) >= pd.to_datetime(_deploy_dates)
n_dropped = (~_keep).sum()
measurement_df = measurement_df[_keep].reset_index(drop=True)
print(f'Dropped {n_dropped} measurement rows before deployment date_min.')

# For geoprecision chain boreholes, loggers were pre-logging before installation.
# Drop measurements before the manually recorded installation time (CEST converted to UTC).
measurement_df['_datetime'] = pd.to_datetime(
    measurement_df['date_min'] + ' ' + measurement_df['time'], errors='coerce'
)
measurement_df['_install_dt'] = measurement_df['label'].map(INSTALL_DATETIMES_UTC)
pre_install_mask = (
    measurement_df['_install_dt'].notna() &
    (measurement_df['_datetime'] < measurement_df['_install_dt'])
)
n_dropped_install = pre_install_mask.sum()
measurement_df = measurement_df[~pre_install_mask].drop(columns=['_datetime', '_install_dt']).reset_index(drop=True)
print(f'Dropped {n_dropped_install} pre-installation measurement rows using manual installation times.')

# Equilibrium flag: false for first 3 days post-installation, true after.
# For geoprecision boreholes use the precise installation datetime (INSTALL_DATETIMES_UTC);
# for NTC boreholes use midnight UTC on the drill date (no precise time available).
_install_dt_all = {
    label: INSTALL_DATETIMES_UTC.get(label, pd.Timestamp(date_str))
    for label, date_str in drill_date_by_label.items()
}
measurement_df['_datetime'] = pd.to_datetime(
    measurement_df['date_min'] + ' ' + measurement_df['time'], errors='coerce'
)
measurement_df['_install_dt'] = measurement_df['label'].map(_install_dt_all)
_equilibrated = (
    measurement_df['_install_dt'].notna() &
    (measurement_df['_datetime'] >= measurement_df['_install_dt'] + pd.Timedelta(days=3))
)
measurement_df['equilibrium'] = _equilibrated.map({True: 'true', False: 'false'})
measurement_df = measurement_df.drop(columns=['_datetime', '_install_dt'])
n_equilibrated = _equilibrated.sum()
print(f'Equilibrium: {n_equilibrated} rows marked true, {(~_equilibrated).sum()} rows marked false.')

# Round temperatures to match actual sensor precision
# Geoprecision chains: 0.0001 °C resolution (4 dp); Tynitag NTC: 0.01 °C (2 dp)
gp_mask = measurement_df['source_family'] == 'geoprecision'
measurement_df.loc[gp_mask, 'temperature'] = measurement_df.loc[gp_mask, 'temperature'].round(4)
measurement_df.loc[~gp_mask, 'temperature'] = measurement_df.loc[~gp_mask, 'temperature'].round(2)

# Update borehole date range from the measurement table, then override both date_min
# and date_max with the drilling date (= installation day). GLENGlat borehole.date_min
# and date_max represent the period of drilling, not the measurement period.
date_span = (
    measurement_df.groupby('borehole_id', as_index=False)
    .agg(date_min=('date_min', 'min'), date_max=('date_max', 'max'))
)
borehole_df = borehole_df.merge(date_span, left_on='id', right_on='borehole_id', how='left', suffixes=('', '_meas'))
borehole_df['date_min'] = borehole_df['date_min_meas'].fillna(borehole_df['date_min'])
borehole_df['date_max'] = borehole_df['date_max_meas'].fillna(borehole_df['date_max'])
borehole_df = borehole_df.drop(columns=['borehole_id', 'date_min_meas', 'date_max_meas'])
borehole_df['date_min'] = borehole_df['label'].map(drill_date_by_label).fillna(borehole_df['date_min'])
borehole_df['date_max'] = borehole_df['label'].map(drill_date_by_label).fillna(borehole_df['date_max'])

measurement_required = ['borehole_id', 'depth', 'temperature', 'date_max']
missing_measurement_required = [col for col in measurement_required if col not in measurement_df.columns]
if missing_measurement_required:
    raise ValueError(f'Missing required measurement values: {missing_measurement_required}')

display(borehole_df)
display(measurement_df.head())
print(f'Built {len(borehole_df)} borehole rows from {len(SOURCE_SPECS)} sensor series.')
print(f'Built {len(measurement_df)} measurement rows.')


/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/processing/thermistor_processing.py:671: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  when_dt = pd.to_datetime(str(when), dayfirst=True, errors='coerce')
/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/processing/thermistor_processing.py:671: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  when_dt = pd.to_datetime(str(when), dayfirst=True, errors='coerce')
/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/processing/thermistor_processing.py:671: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silenc

Dropped 4220 non-hourly geoprecision measurement rows.
Dropped 17590 measurement rows before deployment date_min.
Dropped 240 pre-installation measurement rows using manual installation times.
Equilibrium: 303563 rows marked true, 12662 rows marked false.


,id,glacier_name,glims_id,latitude,longitude,elevation,mass_balance_area,label,date_min,date_max,drill_method,ice_depth,depth,to_bed,temperature_uncertainty,notes,investigators,funding
0,1,Alphubel South,G007867E46057N,46.056151,7.867965,3802.0,accumulation,AH1G,2025-08-05,2025-08-05,thermal,0,50.60,True,0.05,Hourly englacial temperature logger (geoprecis...,Janosch Beer (ORCID: 0009-0002-1128-5470; VAW ...,ETH Zurich; Centre de recherche sur l'environn...
1,2,Alphubel South,G007867E46057N,46.055683,7.867402,3759.0,ablation,AH2G,2025-08-06,2025-08-06,thermal,0,20.30,True,0.05,Hourly englacial temperature logger (geoprecis...,Janosch Beer (ORCID: 0009-0002-1128-5470; VAW ...,ETH Zurich; Centre de recherche sur l'environn...
2,3,Alphubel South,G007867E46057N,46.056422,7.868204,3820.0,accumulation,AH3G,2025-08-06,2025-08-06,thermal,0,58.30,True,0.05,Hourly englacial temperature logger (geoprecis...,Janosch Beer (ORCID: 0009-0002-1128-5470; VAW ...,ETH Zurich; Centre de recherche sur l'environn...
3,4,Hohsaasgletscher,G008002E46138N,46.138464,8.002132,3349.0,ablation,HS1G,2025-08-13,2025-08-13,thermal,0,29.00,True,0.05,Hourly englacial temperature logger (geoprecis...,Janosch Beer (ORCID: 0009-0002-1128-5470; VAW ...,ETH Zurich; Centre de recherche sur l'environn...
4,5,Hohsaasgletscher,G008002E46138N,46.137737,7.999895,3278.0,ablation,HS2G,2025-08-13,2025-08-13,thermal,0,21.50,True,0.05,Hourly englacial temperature logger (geoprecis...,Janosch Beer (ORCID: 0009-0002-1128-5470; VAW ...,ETH Zurich; Centre de recherche sur l'environn...
5,6,Hohsaasgletscher,G008002E46138N,46.137687,8.000518,3292.0,ablation,HS3G,2025-08-13,2025-08-13,thermal,0,21.50,False,0.05,Hourly englacial temperature logger (geoprecis...,Janosch Beer (ORCID: 0009-0002-1128-5470; VAW ...,ETH Zurich; Centre de recherche sur l'environn...
6,7,Chessjengletscher,G007919E46068N,46.066094,7.916425,3031.0,ablation,CJ1G,2025-08-07,2025-08-07,thermal,0,38.30,True,0.05,Hourly englacial temperature logger (geoprecis...,Janosch Beer (ORCID: 0009-0002-1128-5470; VAW ...,ETH Zurich; Centre de recherche sur l'environn...
7,8,Chessjengletscher,G007919E46068N,46.066735,7.916336,3005.0,ablation,CJ2G,2025-08-07,2025-08-07,thermal,0,17.00,True,0.05,Hourly englacial temperature logger (geoprecis...,Janosch Beer (ORCID: 0009-0002-1128-5470; VAW ...,ETH Zurich; Centre de recherche sur l'environn...
8,9,Alphubel South,G007867E46057N,46.055553,7.868256,3775.0,ablation,AH1TT,2024-08-21,2024-08-21,thermal,0,15.00,False,0.20,Hourly englacial temperature logger (tynitag);...,Janosch Beer (ORCID: 0009-0002-1128-5470; VAW ...,ETH Zurich; Centre de recherche sur l'environn...
9,10,Alphubel South,G007867E46057N,46.055791,7.868283,3789.0,ablation,AH2TT,2024-08-21,2024-08-21,thermal,0,14.00,False,0.20,Hourly englacial temperature logger (tynitag);...,Janosch Beer (ORCID: 0009-0002-1128-5470; VAW ...,ETH Zurich; Centre de recherche sur l'environn...


,borehole_id,depth,temperature,date_min,date_max,time,utc_offset,equilibrium,notes,source_family,sensor_name,label
0,1,18.0,0.0040,2025-08-05,2025-08-05,15:00:00,0,false,Hourly Geoprecision chain data from A551FE_202...,geoprecision,#1,AH1G
1,1,21.0,-0.2243,2025-08-05,2025-08-05,15:00:00,0,false,Hourly Geoprecision chain data from A551FE_202...,geoprecision,#2,AH1G
2,1,24.0,-0.0445,2025-08-05,2025-08-05,15:00:00,0,false,Hourly Geoprecision chain data from A551FE_202...,geoprecision,#3,AH1G
3,1,27.0,0.0745,2025-08-05,2025-08-05,15:00:00,0,false,Hourly Geoprecision chain data from A551FE_202...,geoprecision,#4,AH1G
4,1,30.0,-0.1156,2025-08-05,2025-08-05,15:00:00,0,false,Hourly Geoprecision chain data from A551FE_202...,geoprecision,#5,AH1G


Built 23 borehole rows from 23 sensor series.
Built 316225 measurement rows.


## Quick QA

Before exporting, the notebook checks the required GLENGlat fields, verifies the borehole references, and summarizes the resulting submission tables.

In [16]:
required_borehole_cols = ['id', 'glacier_name', 'latitude', 'longitude', 'elevation', 'mass_balance_area', 'label', 'depth']
required_measurement_cols = ['borehole_id', 'depth', 'temperature', 'date_max']

missing_borehole_cols = [col for col in required_borehole_cols if col not in borehole_df.columns]
missing_measurement_cols = [col for col in required_measurement_cols if col not in measurement_df.columns]
if missing_borehole_cols:
    raise ValueError(f'Missing borehole columns: {missing_borehole_cols}')
if missing_measurement_cols:
    raise ValueError(f'Missing measurement columns: {missing_measurement_cols}')

valid_mass_balance = {'ablation', 'accumulation'}
invalid_mass_balance = sorted(set(borehole_df['mass_balance_area'].dropna()) - valid_mass_balance)
if invalid_mass_balance:
    raise ValueError(f'Invalid mass_balance_area values: {invalid_mass_balance}')

if measurement_df['borehole_id'].isna().any():
    raise ValueError('Some measurement rows do not resolve to a borehole_id.')

orphan_ids = sorted(set(measurement_df['borehole_id']) - set(borehole_df['id']))
if orphan_ids:
    raise ValueError(f'measurement.borehole_id references missing borehole.id: {orphan_ids}')

print('Boreholes:', len(borehole_df))
print('Measurements:', len(measurement_df))
print('Mass-balance areas:', borehole_df['mass_balance_area'].value_counts(dropna=False).to_dict())
print('Borehole depth range [m]:', borehole_df['depth'].min(), 'to', borehole_df['depth'].max())
print('Measurement temperature range [degC]:', measurement_df['temperature'].min(), 'to', measurement_df['temperature'].max())

display(
    measurement_df.groupby('borehole_id', as_index=False)
    .agg(n_rows=('temperature', 'size'), depth_min=('depth', 'min'), depth_max=('depth', 'max'))
    .sort_values('borehole_id')
)

Boreholes: 23
Measurements: 316225
Mass-balance areas: {'ablation': 21, 'accumulation': 2}
Borehole depth range [m]: 7.0 to 58.3
Measurement temperature range [degC]: -6.92 to 33.7503


,borehole_id,n_rows,depth_min,depth_max
0,1,10010,18.00,45.00
1,2,4905,8.30,20.30
2,3,3860,20.00,40.00
3,4,10860,2.00,29.00
4,5,5430,1.50,21.50
5,6,5425,0.20,21.50
6,7,31200,11.30,38.30
7,8,15590,7.00,17.00
8,9,8526,10.00,15.00
9,10,18764,9.00,14.00


## Export submission tables

The output is written to a dated folder under `new_data/glenglat_submission/` so each submission attempt stays separate.

In [17]:
submission_dir = output_dir / pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
submission_dir.mkdir(parents=True, exist_ok=True)

borehole_out = submission_dir / 'borehole.csv'
measurement_out = submission_dir / 'measurement.csv'

borehole_df.to_csv(borehole_out, index=False)
measurement_df.to_csv(measurement_out, index=False)

print('Wrote:', borehole_out)
print('Wrote:', measurement_out)

Wrote: /Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/new_data/glenglat_submission/20260512_161357/borehole.csv
Wrote: /Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/new_data/glenglat_submission/20260512_161357/measurement.csv


## Validate with GLENGlat

The official GLENGlat workflow validates the CSV files with the repository's `submission/validate.py` script and Frictionless. Install `frictionless~=5.13` if you want the notebook to run the extra local check as well.

In [18]:
glenglat_repo = project_root / 'external' / 'glenglat'
glenglat_repo.parent.mkdir(parents=True, exist_ok=True)

if not glenglat_repo.exists():
    print('Cloning GLENGlat repository...')
    clone_cmd = ['git', 'clone', 'https://github.com/mjacqu/glenglat.git', str(glenglat_repo)]
    clone_result = subprocess.run(clone_cmd, capture_output=True, text=True)
    print(clone_result.stdout)
    if clone_result.returncode != 0:
        print(clone_result.stderr)
        raise RuntimeError('Could not clone the GLENGlat repository.')

# validate.py imports frictionless directly, so ensure it exists in this Python env.
try:
    import frictionless  # noqa: F401
except ImportError:
    print('Installing frictionless~=5.13 ...')
    install_cmd = [sys.executable, '-m', 'pip', 'install', 'frictionless~=5.13']
    install_result = subprocess.run(install_cmd, capture_output=True, text=True)
    print(install_result.stdout if install_result.stdout else '(pip stdout empty)')
    if install_result.returncode != 0:
        print(install_result.stderr if install_result.stderr else '(pip stderr empty)')
        raise RuntimeError('Could not install frictionless~=5.13 in the active Python environment.')

validate_script = glenglat_repo / 'submission' / 'validate.py'
if not validate_script.exists():
    raise FileNotFoundError(f'Validation script not found: {validate_script}')

validate_cmd = [sys.executable, str(validate_script), str(submission_dir)]
print('Running:', ' '.join(validate_cmd))
validate_result = subprocess.run(validate_cmd, capture_output=True, text=True)
print('--- STDOUT ---')
print(validate_result.stdout if validate_result.stdout else '(none)')
print('--- STDERR ---')
print(validate_result.stderr if validate_result.stderr else '(none)')
print('Exit code:', validate_result.returncode)

if validate_result.returncode != 0:
    raise RuntimeError('GLENGlat validation failed.')

Running: /Users/janoschbeer/.julia/conda/3/x86_64/envs/polythermal_swiss_glaciers/bin/python /Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/external/glenglat/submission/validate.py /Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/new_data/glenglat_submission/20260512_161357
--- STDOUT ---

# -----
# valid: borehole.csv 
# -----

## Summary 

+--------------+---------------+
| Name         | Value         |
+==============+===============+
| File Place   | borehole.csv  |
+--------------+---------------+
| File Size    | 13.2 kB       |
+--------------+---------------+
| Total Time   | 0.088 Seconds |
+--------------+---------------+
| Rows Checked | 23            |
+--------------+---------------+


# -----
# valid: measurement.csv 
# -----

## Summary 

+--------------+-----------------+
| Name         | Value           |
+==============+=================+
| File Place   | m

try:
    from frictionless import validate
except ImportError:
    print('frictionless is not installed in this environment.')
    print('Install with: pip install "frictionless~=5.13"')
else:
    borehole_report = validate(str(borehole_out))
    measurement_report = validate(str(measurement_out))
    print('borehole.csv valid:', borehole_report.valid)
    print('measurement.csv valid:', measurement_report.valid)
    if not borehole_report.valid:
        print(borehole_report.flatten(['type', 'message', 'rowNumber', 'fieldName']))
    if not measurement_report.valid:
        print(measurement_report.flatten(['type', 'message', 'rowNumber', 'fieldName']))

In [7]:
try:
    from frictionless import validate
except ImportError:
    print('frictionless is not installed in this environment.')
    print('Install with: pip install "frictionless~=5.13"')
else:
    borehole_report = validate(str(borehole_out))
    measurement_report = validate(str(measurement_out))
    print('borehole.csv valid:', borehole_report.valid)
    print('measurement.csv valid:', measurement_report.valid)
    if not borehole_report.valid:
        print(borehole_report.flatten(['type', 'message', 'rowNumber', 'fieldName']))
    if not measurement_report.valid:
        print(measurement_report.flatten(['type', 'message', 'rowNumber', 'fieldName']))

borehole.csv valid: False
measurement.csv valid: False
[['resource-error', 'The data resource has an error: path "/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/new_data/glenglat_submission/20260512_115521/borehole.csv" is not safe', None, None]]
[['resource-error', 'The data resource has an error: path "/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/new_data/glenglat_submission/20260512_115521/measurement.csv" is not safe', None, None]]
